In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD

# ------------------------------------------------------------
# Formatting helpers (2 decimals, fixed width) + FIXED COLUMN WIDTHS
# ------------------------------------------------------------
np.set_printoptions(precision=2, suppress=True, floatmode="fixed")

def format_matrix_with_mask(M, mask=None, width=5):
    """
    Format matrix rows with fixed-width 2-decimal numbers.
    If mask is provided (same shape), masked-out entries print as blanks.
    """
    out = []
    for r in range(M.shape[0]):
        parts = []
        for c in range(M.shape[1]):
            if mask is not None and not mask[r, c]:
                parts.append(" " * width)
            else:
                parts.append(f"{M[r, c]:{width}.2f}")
        out.append(" ".join(parts))
    return out

def blank_rows(n_rows, n_cols, width=5):
    row = " ".join(" " * width for _ in range(n_cols))
    return [row for _ in range(n_rows)]

def print_side_by_side_fixed(titles, matrices, shapes, masks=None, spacer=" | ", width=5):
    """
    Print matrices side-by-side with FIXED reserved widths/heights so alignment
    does NOT shift as k changes.

    - shapes: fixed (rows, cols) reserved for each block
    - masks:  optional list of boolean masks (same shape as each matrix block),
              where False entries are printed as blanks (not zeros)
    """
    if masks is None:
        masks = [None] * len(matrices)

    formatted = []
    for M, mk, (r_fix, c_fix) in zip(matrices, masks, shapes):
        assert M.shape == (r_fix, c_fix), "Each matrix must already match its fixed shape."
        if mk is not None:
            assert mk.shape == M.shape, "Mask must match matrix shape."
        formatted.append(format_matrix_with_mask(M, mask=mk, width=width))

    max_h = max(r for r, c in shapes)

    # pad each block to max_h (usually only affects blocks with fewer reserved rows)
    padded_blocks = []
    for (r_fix, c_fix), rows in zip(shapes, formatted):
        if r_fix < max_h:
            rows = rows + blank_rows(max_h - r_fix, c_fix, width=width)
        padded_blocks.append(rows)

    # fixed character width per title block
    block_char_widths = []
    for (r_fix, c_fix) in shapes:
        block_char_widths.append(c_fix * width + (c_fix - 1) * 1 if c_fix > 0 else 0)

    title_line = spacer.join(f"{t:^{w}}" for t, w in zip(titles, block_char_widths))
    print(title_line)
    print("-" * len(title_line))

    for r in range(max_h):
        print(spacer.join(padded_blocks[i][r] for i in range(len(padded_blocks))))
    print()

In [7]:
# ------------------------------------------------------------
# Random integer matrix
# ------------------------------------------------------------
np.random.seed(0)
m, n = 4, 3
A = np.random.randint(-5, 6, size=(m, n)).astype(float)
rank = np.linalg.matrix_rank(A)

# ------------------------------------------------------------
# Fixed shapes reserved for each block (so nothing shifts)
# ------------------------------------------------------------
U_shape  = (m, rank)
S_shape  = (rank, rank)
Vt_shape = (rank, n)
A_shape  = (m, n)

# ------------------------------------------------------------
# Truncated SVD for k = 1 ... rank
# ------------------------------------------------------------
for k in range(1, rank + 1):
    svd = TruncatedSVD(n_components=k, random_state=0)
    US = svd.fit_transform(A)        # m x k  (U Σ)
    Vt = svd.components_             # k x n  (V^T)
    s  = svd.singular_values_        # (k,)

    S = np.diag(s)                   # k x k
    U = US @ np.linalg.inv(S)        # m x k
    A_recon = US @ Vt                # m x n

    # Allocate fixed-shape blocks (values don't matter where masked out)
    U_block  = np.zeros(U_shape)
    S_block  = np.zeros(S_shape)
    Vt_block = np.zeros(Vt_shape)

    # Place the actual values into the top-left / leading parts
    U_block[:, :k] = U
    S_block[:k, :k] = S
    Vt_block[:k, :] = Vt

    # Masks: True where entries are "real", False where we want blanks
    U_mask  = np.zeros(U_shape, dtype=bool);   U_mask[:, :k] = True
    S_mask  = np.zeros(S_shape, dtype=bool);   S_mask[:k, :k] = True
    Vt_mask = np.zeros(Vt_shape, dtype=bool);  Vt_mask[:k, :] = True

    print("=" * 120)
    print(f"k = {k}")
    print("=" * 120)

    print_side_by_side_fixed(
        titles=["U", "S", "Vᵀ", "A_reconstructed", "A_original"],
        matrices=[U_block, S_block, Vt_block, A_recon, A],
        shapes=[U_shape, S_shape, Vt_shape, A_shape, A_shape],
        masks=[U_mask, S_mask, Vt_mask, None, None],
        spacer=" | ",
        width=5,
    )

k = 1
        U         |         S         |        Vᵀ         |  A_reconstructed  |    A_original    
-------------------------------------------------------------------------------------------------
-0.69             |  7.25             | -0.15  0.73  0.67 |  0.73 -3.64 -3.33 |  0.00 -5.00 -2.00
 0.61             |                   |                   | -0.65  3.23  2.95 | -2.00  2.00  4.00
-0.24             |                   |                   |  0.25 -1.25 -1.14 | -2.00  0.00 -3.00
 0.31             |                   |                   | -0.33  1.66  1.52 | -1.00  2.00  1.00

k = 2
        U         |         S         |        Vᵀ         |  A_reconstructed  |    A_original    
-------------------------------------------------------------------------------------------------
-0.69  0.41       |  7.25  0.00       | -0.15  0.73  0.67 |  1.31 -4.46 -2.30 |  0.00 -5.00 -2.00
 0.61  0.26       |  0.00  3.50       |  0.40 -0.57  0.72 | -0.28  2.70  3.61 | -2.00  2.00  4.00
-0.24 -